# Dependencies

In [33]:
import pickle
import numpy as np

from itertools import groupby
from scipy.stats import mode
from collections import defaultdict

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans, KMeans

import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

import umap

In [34]:
L = 4500 # original length
tp = 1
l = int(L/tp) # after patch
N = 20
D = 192
sample_frequency = 9

fmr1_fold_1 = {"train":[402, 404, 405, 406, 407, 408], "valid": [401, 403]}
fmr1_fold_2 = {"train":[401, 403, 405, 406, 407, 408], "valid": [402, 404]}
fmr1_fold_3 = {"train":[401, 402, 403, 404, 407, 408], "valid": [405, 406]}
fmr1_fold_4 = {"train":[401, 402, 404, 405, 406, 407], "valid": [403, 408]}

# MAE representations for Behavior classification

## representations

In [41]:
# Load skeletonMAE representation
#tr_feats = np.load('/home/rguo_hpc/myfolder/mocap/outputs/representations/fmr1/t3/representations/mae_sdannce_tr.npy')[:, 40:1540][:,::sample_frequency]
#val_feats = np.load('/home/rguo_hpc/myfolder/mocap/outputs/representations/fmr1/t3/representations/mae_sdannce_val.npy')[:, 40:1540][:,::sample_frequency]
tr_feats = np.load("/home/rguo_hpc/myfolder/mocap/outputs/fmr1/250/representations/mae_sdannce_tr.npy")[:, 125:4625][:,::sample_frequency]
val_feats = np.load("/home/rguo_hpc/myfolder/mocap/outputs/fmr1/250/representations/mae_sdannce_val.npy")[:, 125:4625][:,::sample_frequency]
feats = np.concatenate([tr_feats, val_feats])
print(tr_feats.shape, val_feats.shape)

(360, 500, 192) (120, 500, 192)


In [ ]:
# Representations learned after mil
mil_feats = np.load('/home/rguo_hpc/myfolder/mocap/outputs/50/mil/representations.npy')[:, 1:]#[:,::sample_frequency]
print(mil_feats.shape)
mil_tr_feats = mil_feats[:tr_feats.shape[0]]
mil_val_feats = mil_feats[tr_feats.shape[0]:]

## mouse, genotype, hlac, llac labels

In [42]:
# load original data 
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    result = pickle.load(file)

for mouse in result.keys():     # result.keys(): [408, 407, 404, 403, 402, 406, 405, 401]
    num_seq = len(result[mouse]["ratgen"])
    ratgen  = int(result[mouse]["ratgen"][0])
    result[mouse]["ratgen"] = [ratgen for i in range(num_seq * N,)]
    ratid = result[mouse]["ratid"][0]
    result[mouse]["ratid"] = [ratid for i in range(num_seq * N,)]
    
    # llac    
    llac = np.array(result[mouse]["llac"])
    llac = np.squeeze(llac, axis=2)
    llac = llac.reshape(-1, L,)
    if tp > 1:
        grouped_llac = llac.reshape(-1, l, tp) # choose the mode label of each patch 
        result[mouse]["llac"] = mode(grouped_llac, axis=2, keepdims=False).mode.astype(np.uint8)
    else:
        result[mouse]["llac"] = llac
    
    # hlac
    hlac = np.array(result[mouse]["hlac"]) # (3, 90000, 1)
    hlac = np.squeeze(hlac, axis=2)
    hlac = hlac.reshape(-1, L,)
    if tp > 1:
        grouped_hlac = hlac.reshape(-1, l, 3)
        result[mouse]["hlac"] = mode(grouped_hlac, axis=2, keepdims=False).mode.astype(np.uint8)
    else:
        result[mouse]["hlac"] = hlac
    
    del result[mouse]["m1"]

In [43]:
# Train: mouse, genotype, hlac, llac
mouse_tr, gen_tr, hlac_tr, llac_tr = [], [], [], []
for mouse_id in fmr1_fold_1["train"]:
    mouse_tr.append(result[mouse_id]["ratid"])
    gen_tr.append(result[mouse_id]["ratgen"])
    hlac_tr.append(result[mouse_id]["hlac"])
    llac_tr.append(result[mouse_id]["llac"])
    
mouse_tr = np.repeat(np.concatenate(mouse_tr), l)[::sample_frequency]
gen_tr =  np.repeat(np.concatenate(gen_tr), l)[::sample_frequency]
hlac_tr = np.concatenate(hlac_tr)[:,::sample_frequency]
llac_tr = np.concatenate(llac_tr)[:,::sample_frequency]

# Val:  mouse, genotype, hlac, llac
mouse_val, gen_val, hlac_val, llac_val = [], [], [], []
for mouse_id in fmr1_fold_1["valid"]:
    mouse_val.append(result[mouse_id]["ratid"])
    gen_val.append(result[mouse_id]["ratgen"])
    hlac_val.append(result[mouse_id]["hlac"])
    llac_val.append(result[mouse_id]["llac"]) 
    
mouse_val = np.repeat(np.concatenate(mouse_val), l)[::sample_frequency]
gen_val = np.repeat(np.concatenate(gen_val), l)[::sample_frequency]
hlac_val = np.concatenate(hlac_val)[:,::sample_frequency]
llac_val = np.concatenate(llac_val)[:,::sample_frequency]

## behavior classification

In [44]:
# Flatten 
tr_feats = tr_feats.reshape(-1, D)
val_feats = val_feats.reshape(-1, D)#[::sample_frequency]

#mil_tr_feats = mil_tr_feats.reshape(-1, D)
#mil_val_feats = mil_val_feats.reshape(-1, D)#[::sample_frequency]

hlac_tr = hlac_tr.reshape(hlac_tr.size, )
hlac_val = hlac_val.reshape(hlac_val.size, )#[::sample_frequency]

llac_tr = llac_tr.reshape(llac_tr.size, )
llac_val = llac_val.reshape(llac_val.size, )#[::sample_frequency]

In [45]:
# skeleton MAE
model = LogisticRegression(max_iter=500, multi_class='multinomial')
#model = RandomForestClassifier()
model.fit(tr_feats, hlac_tr)
# Predict
y_pred = model.predict(val_feats)

# Evaluate
print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))

/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.675

Classification Report:
               precision    recall  f1-score   support

           1       0.69      0.47      0.56     11792
           2       0.59      0.69      0.64     16949
           3       0.72      0.82      0.77      2183
           4       0.73      0.68      0.70      2158
           5       0.44      0.33      0.38       961
           6       0.91      0.94      0.93      6165
           7       0.63      0.68      0.66     12772
           8       0.75      0.75      0.75      7020

    accuracy                           0.68     60000
   macro avg       0.68      0.67      0.67     60000
weighted avg       0.68      0.68      0.67     60000



/home/rguo_hpc/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Mil features
mil_model = LogisticRegression(max_iter=500, multi_class='multinomial')
#mil_model = RandomForestClassifier()
mil_model.fit(mil_tr_feats, hlac_tr)
# Predict
y_pred = mil_model.predict(mil_val_feats)

# Evaluate
print("Accuracy:", accuracy_score(hlac_val, y_pred))
print("\nClassification Report:\n", classification_report(hlac_val, y_pred))

# Behavior distribution 
### Load data

In [ ]:
# Load data
with open("/home/rguo_hpc/myfolder/data/sdannce/data_fmr1.pkl", 'rb') as file:
    result = pickle.load(file)

# mouse
mice = result.keys()
print(mice)

In [ ]:
# collect all hlac labels of all frames
hlac_original = []
llac_original = []
gen_original = []

for mouse in mice: 
    # a list of genotype labels
    num_seq = len(result[mouse]["ratgen"])
    ratgen  = int(result[mouse]["ratgen"][0])
    gen_original += [ratgen for i in range(num_seq * N,)]
    
    # llac behaviors label
    llac = np.array(result[mouse]["llac"])
    llac = np.squeeze(llac, axis=2)
    llac_original.append(llac.reshape(-1, L, ))
    
    # hlac behaviors label
    hlac = np.array(result[mouse]["hlac"]) # (3, 90000, 1)
    hlac = np.squeeze(hlac, axis=2)
    hlac_original.append(hlac.reshape(-1, L, ))

# all hlac, llac labels
hlac_original = np.concatenate(hlac_original) #  (n, L) (480, 4500)
llac_original = np.concatenate(llac_original)

In [ ]:
# number of sequences
n = len(hlac_original)

# Split genotype labels to train & test
gen_tr  = gen_original[:180] + gen_original[240:420]
gen_val = gen_original[180:240] + gen_original[420:]

# indices of genotype
indices_1 = [i for i, v in enumerate(gen_original) if v == 1] # 300
indices_0 = [i for i, v in enumerate(gen_original) if v == 0] # 180

### Summary of behavior length

In [ ]:
"""
# Define function to summary behaviors length
def run_length_encoding(seq):
    change = np.flatnonzero(np.diff(seq)) + 1        # indices where value changes
    lengths = np.diff(np.concatenate(([0], change, [seq.shape[0]]))) # run lengths
    values = seq[np.concatenate(([0], change))]      # corresponding values
    return values, lengths

# Summary behaviors length
# hlac
all_values_hlac, all_lengths_hlac = [], []
length_dist_hlac = defaultdict(list)
for i in range(n):
    seq = hlac_original[i]
    values, lengths = run_length_encoding(seq)
    all_values_hlac.append(values)
    all_lengths_hlac.append(lengths)
"""

In [ ]:
# Transition matrix side by side for each label
n_states = 9
n_states_l = 162

def get_runs(seq):
    """Return list of (state, run_length) for one sequence."""
    return [(state, len(list(group))) for state, group in groupby(seq)]

all_runs = [get_runs(seq) for seq in hlac_original]  # list of 480 lists of (state, length)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, indices, label in zip(axes, [indices_0, indices_1[:180]], [0, 1]):
    trans_counts = np.zeros((n_states, n_states))
    for idx in indices:
        runs = all_runs[idx]
        states = [s for s, _ in runs]  # collapsed sequence of states (no repeats)
        for i in range(len(states) - 1):
            from_s, to_s = states[i] - 1, states[i + 1] - 1  # 0-indexed
            trans_counts[from_s, to_s] += 1

    row_sums = trans_counts.sum(axis=1, keepdims=True)
    trans_probs = np.divide(trans_counts, row_sums, out=np.zeros_like(trans_counts), where=row_sums != 0)

    im = ax.imshow(trans_probs, cmap='viridis', vmin=0, vmax=trans_probs.max())
    ax.set_xticks(range(n_states))
    ax.set_yticks(range(n_states))
    ax.set_xticklabels(range(1, n_states + 1))
    ax.set_yticklabels(range(1, n_states + 1))
    ax.set_xlabel("To state")
    ax.set_ylabel("From state")
    ax.set_title(f"Category {label} (P(to | from), changes only)")

    for i in range(n_states):
        for j in range(n_states):
            if trans_probs[i, j] > 0:
                color = 'white' if trans_probs[i, j] < trans_probs.max() * 0.6 else 'black'
                ax.text(j, i, f"{trans_probs[i, j]:.2f}", ha='center', va='center', color=color, fontsize=8)

    fig.colorbar(im, ax=ax, label="Transition probability", fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
lengths_per_state = {s: [] for s in range(1, n_states + 1)}
for runs in all_runs:
    for state, length in runs:
        lengths_per_state[state].append(length)
print(len(lengths_per_state[1]))

In [ ]:
keys_map = ["idle", "sniff/head","groom", "scrunched", "active crouched", "reared", "explore", "locomotion", "fast/error"]

lengths_per_state_0 = {s: [] for s in range(1, n_states + 1)}
for i in indices_0:
    for state, length in all_runs[i]:
        lengths_per_state_0[state].append(length)

lengths_per_state_1 = {s: [] for s in range(1, n_states + 1)}
for i in indices_1:
    for state, length in all_runs[i]:
        lengths_per_state_1[state].append(length)

### Smooth behaviors

In [ ]:
def filter_short_runs(arr, min_len=5):
    """
    For each row, keep only elements that belong to a run  of consecutive identical values with length >= min_len.
    Returns a list of 1D numpy arrays (ragged, since row lengths shrink).
    """
    result = []
    for row in arr:
        kept = []
        for val, group in groupby(row):
            run = list(group)
            if len(run) >= min_len:
                kept.extend(run)
            # else: drop them entirely
        result.append(np.array(kept, dtype=arr.dtype))
    return result

# smooth hlac_original
hlac_original_smooth = filter_short_runs(hlac_original, min_len=5)# remove behaviors
all_runs_smooth = [get_runs(seq) for seq in hlac_original_smooth]  

lengths_per_state_0_smooth = {s: [] for s in range(1, n_states + 1)}
for i in indices_0:
    for state, length in all_runs_smooth[i]:
        lengths_per_state_0_smooth[state].append(length)

lengths_per_state_1_smooth = {s: [] for s in range(1, n_states + 1)}
for i in indices_1:
    for state, length in all_runs_smooth[i]:
        lengths_per_state_1_smooth[state].append(length)

### visualization

In [ ]:
def _trim_outliers(values, keep_pct=99):#Keep only `keep_pct`% of values (trim asymmetric tails)
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values
    tail = (100 - keep_pct) / 4 *3
    lo, hi = np.percentile(values, [tail, 100 - tail])
    return values[(values >= lo) & (values <= hi)]
  
def compare_dict_histograms(dict1, dict2, label1="genotype 0", label2="genotype 1", bins=20, keys=range(1, 10),
                            density=True, keep_pct=99, sharex=False, sharey=False):
    """
    Plot side-by-side (grouped) bar histograms comparing two dicts (key -> list of numbers) across the given keys, in a 3x3 grid.
    ----------
    keys : iterable of keys to plot (default 1..9)
    density : bool, normalize histograms so they're comparable. even if the two lists have different lengths
    keep_pct : float, percentage of central values to keep per key (trims extreme outliers symmetrically). Set to 100 to disable trimming.
    sharex, sharey : bool, whether subplots share axes
    """
    keys = list(keys)
    n = len(keys)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), sharex=sharex, sharey=sharey)
    axes = np.array(axes).reshape(-1)

    for i, k in enumerate(keys):
        ax = axes[i]
        v1_raw = dict1.get(k, [])
        v2_raw = dict2.get(k, [])
        if len(v1_raw) == 0 and len(v2_raw) == 0:
            ax.set_title(f"Key {keys_map[i]} (no data)")
            ax.axis("off")
            continue
            
        if k==9:
            v1 = np.array(v1_raw)
            v2 = np.array(v2_raw)
        else: 
            # Trim outliers independently per list (each keeps its own central keep_pct%)
            v1 = _trim_outliers(v1_raw, keep_pct) if len(v1_raw) else np.array([])
            v2 = _trim_outliers(v2_raw, keep_pct) if len(v2_raw) else np.array([])
        
        all_vals = np.concatenate([v1, v2]) if len(v1) or len(v2) else np.array([0, 1])
        lo, hi = all_vals.min(), all_vals.max()
        if lo == hi:
            lo, hi = lo - 0.5, hi + 0.5
        bin_edges = np.linspace(lo, hi, bins + 1)
        bin_width = bin_edges[1] - bin_edges[0]
        centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        counts1, _ = np.histogram(v1, bins=bin_edges, density=density) if len(v1) else (np.zeros(bins), None)
        counts2, _ = np.histogram(v2, bins=bin_edges, density=density) if len(v2) else (np.zeros(bins), None)

        offset = bin_width * 0.2
        ax.bar(centers - offset, counts1, width=bin_width * 0.4, label=label1, color="tab:blue", edgecolor="white")
        ax.bar(centers + offset, counts2, width=bin_width * 0.4, label=label2, color="tab:orange", edgecolor="white")
        # Calculate mean
        if len(v1):
            mean1 = np.mean(v1)
            ax.axvline(mean1, color="tab:blue", linestyle="--", linewidth=1.5, label=f"{label1} mean={mean1:.2f}")
        if len(v2):
            mean2 = np.mean(v2)
            ax.axvline(mean2, color="tab:orange", linestyle="--", linewidth=1.5, label=f"{label2} mean={mean2:.2f}")
        
        ax.set_title(f"State {k} {keys_map[i]}  (n1={len(v1_raw)}, n2={len(v2_raw)})")
        ax.legend(fontsize=8)

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"Distribution comparison across keys (central {keep_pct}% shown)", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
compare_dict_histograms(lengths_per_state_0_smooth, lengths_per_state_1_smooth, keep_pct=98.8,  bins=20)

In [ ]:
compare_dict_histograms(lengths_per_state_0, lengths_per_state_1, keep_pct=98.8,  bins=20)

## Histogram

In [ ]:
# Create histograms of hlac, llac 
histograms_hlac = np.stack([np.bincount(hlac_original[i], minlength=10)[1:9] for i in range(n)])
histograms_llac = np.array([np.bincount(llac_original[i], minlength=163)[1:162] for i in range(n)])

# Create train & test histogram for hla
histograms_hlac_tr = np.concatenate([histograms_hlac[:180], histograms_hlac[240:420]])
histograms_hlac_val = np.concatenate([histograms_hlac[180:240], histograms_hlac[420:]])
histograms_hlac_tr = histograms_hlac_tr /histograms_hlac_tr.sum(axis=1, keepdims=True)
histograms_hlac_val = histograms_hlac_val /histograms_hlac_val.sum(axis=1, keepdims=True)

# Create train & test histogram for llac
histograms_llac_tr = np.concatenate([histograms_llac[:180], histograms_llac[240:420]])
histograms_llac_val = np.concatenate([histograms_llac[180:240], histograms_llac[420:]])
histograms_llac_tr = histograms_llac_tr /histograms_llac_tr.sum(axis=1, keepdims=True)
histograms_llac_val = histograms_llac_val /histograms_llac_val.sum(axis=1, keepdims=True)

### Genotype classification with behavior histogram features 

In [ ]:
model = RandomForestClassifier()
#model = LogisticRegression()
# hlac
model.fit(histograms_hlac_tr, gen_tr)
# Predict
y_pred = model.predict(histograms_hlac_val)
print("Accuracy:", accuracy_score(gen_val, y_pred))
print("\nClassification Report:\n", classification_report(gen_val, y_pred))

In [ ]:
model = RandomForestClassifier()
#model = LogisticRegression()
# llac
model.fit(histograms_llac_tr, gen_tr)
# Predict 
y_pred = model.predict(histograms_llac_val)
print("Accuracy:", accuracy_score(gen_val, y_pred))
print("\nClassification Report:\n", classification_report(gen_val, y_pred))

### hlac cluster

In [ ]:
N_CLUSTERS = 4   # set an int to fix cluster count, or leave None to auto-select via BIC
RANDOM_STATE = 42
X = histograms_llac

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, max_iter=50)
combined_labels = kmeans.fit_predict(X)

#sil = silhouette_score(X , cluster_labels)
#print(f"Silhouette score: {sil:.3f}")

# UMAP
reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=RANDOM_STATE)
proj = reducer.fit_transform(X)   # (480, 2)
y = gen_original

In [ ]:
#combined_labels = np.array(gen_original)
half_n = len(proj) // 2    # Assuming 'proj' (n, 2) and 'labels' (n,) are already defined
proj_pre, labels_pre = proj[indices_0],  combined_labels[indices_0]
proj_down, labels_down = proj[indices_1][120:],  combined_labels[indices_1][120:]

# shared axis limits across all 3 panels so they're directly comparable
pad = 0.08 * (proj[:, 0].max() - proj[:, 0].min())
xlim = (proj[:, 0].min() - pad, proj[:, 0].max() + pad)
ylim = (proj[:, 1].min() - pad, proj[:, 1].max() + pad)
cmap = get_cmap('tab20', len(set(combined_labels)))
 
plt.rcParams["axes.edgecolor"] = "#444444"
plt.rcParams["axes.linewidth"] = 0.8
 
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharex=True, sharey=True) 
panel_data = [(proj[:, 0], proj[:, 1], combined_labels, "All sequences"),
              (proj_pre[:, 0], proj_pre[:, 1], labels_pre,  " Genotype 0"),
              (proj_down[:, 0], proj_down[:, 1], labels_down,  "Genotype 1"),]
vmin, vmax = combined_labels.min(), combined_labels.max()

for ax, (x, y, labels, short_title) in zip(axes, panel_data):
    ax.scatter(x, y, c=labels, cmap=cmap, s=22, alpha=0.85, linewidths=0.4, edgecolors="white", vmin=vmin, vmax=vmax,)
    ax.set_title(short_title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor("#fafafa")
  
plt.tight_layout()
plt.show()

### hlac distribution

In [ ]:
values = np.arange(1, 10)  # 1-9

# Count occurrences of each value, per category
counts_0 = np.array([(hlac_original[indices_0] == v).sum() for v in values])
counts_1 = np.array([(hlac_original[indices_1][120:] == v).sum() for v in values])
# Normalize to proportions so category sizes don't skew comparison
props_0 = counts_0 / counts_0.sum()
props_1 = counts_1 / counts_1.sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# --- Side-by-side bar chart ---
ax = axes[0]
width = 0.35
x = np.arange(len(values))
ax.bar(x - width/2, props_0, width, label='Category 0', color='#4C72B0', alpha=0.85)
ax.bar(x + width/2, props_1, width, label='Category 1', color='#DD8452', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(values)
ax.set_ylabel("Proportion")
ax.set_title("Value distribution by category")
ax.legend(frameon=False)
ax.grid(alpha=0.25, axis='y')
ax.spines[['top', 'right']].set_visible(False)

# --- Difference plot (highlights which values differ most) ---
ax2 = axes[1]
diff = props_0 - props_1
colors = ['#4C72B0' if d > 0 else '#DD8452' for d in diff]
ax2.bar(x, diff, color=colors, alpha=0.85)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(values)
ax2.set_ylabel("Proportion diff (Cat 0 − Cat 1)")
ax2.set_title("Difference between categories")
ax2.grid(alpha=0.25, axis='y')
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, optimal_leaf_ordering, leaves_list
from scipy.spatial.distance import pdist

def reorder_rows_olo(data, metric='euclidean', method='average'):
    D = pdist(data, metric=metric)      # Pairwise distances between rows
    Z = linkage(D, method=method)       # Hierarchical clustering
    Z_opt = optimal_leaf_ordering(Z, D) # Optimal leaf ordering
    order = leaves_list(Z_opt)          # Final ordering
    return data[order], order


# Reorder each category independently
hist0 = histograms_hlac[indices_0]
hist1 = histograms_hlac[indices_1][120:]

hist0_sorted, order0 = reorder_rows_olo(hist0)
hist1_sorted, order1 = reorder_rows_olo(hist1)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
vmax = max(hist0_sorted.max(), hist1_sorted.max())

im0 = axes[0].imshow(hist0_sorted, aspect='auto',interpolation='nearest',cmap='viridis', vmin=0, vmax=vmax)
axes[0].set_title("Category 0")
axes[0].set_xlabel("State")
axes[0].set_ylabel("Sequences (clustered)")
axes[0].set_xticks(range(hist0.shape[1]))

im1 = axes[1].imshow(hist1_sorted, aspect='auto',interpolation='nearest',cmap='viridis', vmin=0, vmax=vmax)
axes[1].set_title("Category 1")
axes[1].set_xlabel("State")
axes[1].set_ylabel("Sequences (clustered)")
axes[1].set_xticks(range(hist1.shape[1]))

fig.colorbar(im1, ax=axes, label='Count', shrink=0.8)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
vmax = max(histograms_hlac[indices_0].max(), histograms_hlac[indices_1].max())
im0 = axes[0].imshow(histograms_hlac[indices_0], aspect='auto', interpolation='nearest',
                      vmin=0, vmax=vmax, cmap='viridis')
axes[0].set_title('Category 0')
axes[0].set_xlabel('State')
axes[0].set_ylabel('Sequence')
axes[0].set_xticks(range(8))

im1 = axes[1].imshow(histograms_hlac[indices_1][120:], aspect='auto', interpolation='nearest',
                      vmin=0, vmax=vmax, cmap='viridis')
axes[1].set_title('Category 1')
axes[1].set_xlabel('State')
axes[1].set_ylabel('Sequence')
axes[1].set_xticks(range(8))

fig.colorbar(im1, ax=axes, label='Count', shrink=0.8)
plt.show()